# Repository API

Demonstrates imports, exports, updates, and graph operations.
All functions accept an optional `collection` parameter (see Section 5).

In [1]:
import gen
import pathlib
import tempfile

# Fixture files bundled with the repo
REPO_ROOT = pathlib.Path(gen.__file__).parents[3]
FX = REPO_ROOT / "fixtures"

# Fresh temporary repo for each run
WORK_DIR = pathlib.Path(tempfile.mkdtemp(prefix="gen-demo-"))
repo = gen.Repository(str(WORK_DIR))
print(f"Repository at: {WORK_DIR}")

Repository at: /var/folders/f8/8zf8xczs0pxf9_vfnlmqlx9h0000gn/T/gen-demo-6607h8po


## 1. Imports

All import functions share the signature `(filename, sample=None, ...)`.

### 1.1 `import_fasta`

Imports one or more sequences from a FASTA file.  `shallow=True` skips storing sequence bytes and is useful for large references.

In [ ]:
# Minimal: only the filename is required
repo.import_fasta(str(FX / "simple.fa"))

# With an explicit sample name
repo.import_fasta(str(FX / "parts.fa"), sample="parts_sample")

repo.import_fasta(str(FX / "multiseq.fa"), sample="multi")

bgs = repo.get_sequence_graphs()
print(f"Sequence graphs after FASTA imports: {len(bgs)}")
for bg in bgs:
    print(f"  {bg.sample_name} / {bg.name}")

### 1.2 `import_gfa`

Imports a sequence graph from a GFA (Graph Fragment Assembly) file, preserving branch structure.

In [ ]:
repo.import_gfa(str(FX / "anderson_promoters.gfa"), sample="gfa_sample")

bgs = repo.get_sequence_graphs()
print(f"Sequence graphs after GFA import: {len(bgs)}")
for bg in bgs:
    print(f"  {bg.name} (sample={bg.sample_name})")

In [4]:
# Visualise the imported GFA graph
bg_gfa = bgs[0]
bg_gfa.plot()

### 1.3 `import_genbank`

Imports a GenBank (.gb) file, including feature annotations.

In [ ]:
repo.import_genbank(str(FX / "puc19.gb"), sample="genbank_sample")

bgs_gb = repo.get_sequence_graphs()
print(f"Sequence graphs after GenBank import: {len(bgs_gb)}")
for bg in bgs_gb:
    print(f"  {bg.name}")

### 1.4 `import_library` / `import_library_files`

`import_library` takes combinatorial part definitions directly as Python objects.
`import_library_files` reads the same information from CSV files on disk.

In [ ]:
# import_library: build parts programmatically
parts_list = [
    [gen.SequencePart("p1", "AAAA"), gen.SequencePart("cds1", "ATGATAA")],
    [gen.SequencePart("p2", "TAAT"), gen.SequencePart("cds2", "ATGTTAA")],
    [gen.SequencePart("p3", "CAAC"), gen.SequencePart("cds3", "ATGCTAA")],
]

repo.import_library("my_library", parts_list)

bgs_lib = repo.get_sequence_graphs()
print(f"Sequence graphs after library import: {len(bgs_lib)}")

In [ ]:
# import_library_files: read parts and layout from CSV files
PARTS_CSV   = FX / "parts.fa"
LAYOUT_CSV  = FX / "combinatorial_design.csv"

repo.import_library_files("csv_library", str(PARTS_CSV), str(LAYOUT_CSV))

bgs_lib2 = repo.get_sequence_graphs()
print(f"Sequence graphs after library-files import: {len(bgs_lib2)}")

---
## 2. Exports

Export functions write a sample to disk.

In [8]:
# Use the "demo" collection (multiseq.fa) as source for all exports
OUT = WORK_DIR / "exports"
OUT.mkdir(exist_ok=True)

### 2.1 `export_fasta`

In [9]:
out_fa = OUT / "demo.fa"
repo.export_fasta(str(out_fa), sample="multi")
print(out_fa.read_text())

>m1
ATGC
>m2
GCAT
>m3
A

Exported to file /var/folders/f8/8zf8xczs0pxf9_vfnlmqlx9h0000gn/T/gen-demo-6607h8po/exports/demo.fa


### 2.2 `export_gfa`

In [10]:
out_gfa = OUT / "demo.gfa"
repo.export_gfa(str(out_gfa), sample="gfa_sample")
lines = out_gfa.read_text().splitlines()
print("\n".join(lines[:5]))
if len(lines) > 5:
    print(f"... ({len(lines) - 5} more lines)")

S	019e6b47e5cf74b2bee2990567381f5c00000000000000000000000000000000.0.1	A
S	019e6b47e5cf74b2bee299355f017c1f00000000000000000000000000000000.0.1	A
S	019e6b47e5cf74b2bee29998220ecd1400000000000000000000000000000000.0.1	A
S	019e6b47e5cf74b2bee299e4110869aa00000000000000000000000000000000.0.1	A
S	019e6b47e5cf74b2bee29a19d7af716d00000000000000000000000000000000.0.1	A
... (78 more lines)


### 2.3 `export_genbank`

In [11]:
out_gb = OUT / "demo.gb"
repo.export_genbank(str(out_gb), sample="genbank_sample")
print(out_gb.read_text()[:800], "...")

LOCUS       sequence-222046-        2686 bp            linear UNK 01-JAN-1970
FEATURES             Location/Qualifiers
     source          1..2686
                     /mol_type="other DNA"
                     /organism="synthetic DNA construct"
     primer_bind     118..137
                     /label="pBR322ori-F"
                     /note="pBR322 origin, forward primer"
     primer_bind     371..388
                     /label="L4440"
                     /note="L4440 vector, forward primer"
     protein_bind    505..526
                     /label="CAP binding site"
                     /bound_moiety="E. coli catabolite activator protein"
                     /note="CAP binding activates transcription in the presence
                     of cAMP."
     promoter        join(541..546, ...


---
## 3. Updates

Updates graft new variants or sequences onto an existing graph, creating a new sample that branches off the original.

### 3.1 `update_with_fasta`

Replaces a region `[start, end)` of a block group with sequences from a FASTA file.

In [ ]:
# Work on a fresh repo so the updates have a clean baseline
update_repo = gen.Repository(str(WORK_DIR / "updates"))
update_repo.import_fasta(str(FX / "simple.fa"), sample="ref")

update_repo.update_with_fasta(
    str(FX / "parts.fa"),
    sample="ref",
    new_sample="fasta_updated",
    region_name="m123:3-10",
)

bgs = update_repo.get_sequence_graphs()
print(f"Sequence graphs after update: {len(bgs)}")
for bg in bgs:
    print(f"  {bg.sample_name} / {bg.name}")

In [13]:
# The updated block group now has multiple paths; show_path highlights the edited one
updated_bg = [bg for bg in bgs if bg.name == "m123"][0]
w = updated_bg.plot()
w.show_path()
w

### 3.2 `update_with_sequence`

Inline variant: replace a region with a literal sequence string.

In [ ]:
update_repo.update_with_sequence(
    "TTTTTTTT",
    sample="ref",
    new_sample="seq_updated",
    region_name="m123:5-15",
)

bg_seq = [bg for bg in update_repo.get_sequence_graphs() if bg.sample_name == "seq_updated"][0]
w = bg_seq.plot()
w.show_path()
w

### 3.3 `update_with_vcf`

Applies variants from a VCF file.  Supports genotype filtering and multi-sample VCFs.

In [ ]:
# Use a dedicated repo so state is clean
vcf_repo = gen.Repository(str(WORK_DIR / "vcf_simple"))
vcf_repo.import_fasta(str(FX / "simple.fa"), sample="ref")

# reference= names the parent sample to update from
# genotype= fixes a specific GT to apply (e.g. "1/1" = hom alt only)
# sample=  selects a VCF sample column for genotype info
vcf_repo.update_with_vcf(
    str(FX / "simple.vcf"),
    reference="ref",
)

bgs = vcf_repo.get_sequence_graphs()
print(f"Sequence graphs after VCF update: {len(bgs)}")
for bg in bgs:
    print(f"  {bg.sample_name} / {bg.name}")

updated_bg = [bg for bg in bgs if bg.sample_name != "ref"][0]
w = updated_bg.plot()
w.show_path()
w

### 3.4 `update_with_gfa`

Merges an additional GFA graph into an existing block group.

In [ ]:
gfa_update_repo = gen.Repository(str(WORK_DIR / "gfa_update"))
gfa_update_repo.import_gfa(str(FX / "simple.gfa"), sample="ref")

gfa_update_repo.update_with_gfa(
    str(FX / "walk.gfa"),
    sample="ref",
    new_sample="gfa_merged",
)

merged_bgs = gfa_update_repo.get_sequence_graphs()
print(f"Sequence graphs after GFA update: {len(merged_bgs)}")
for bg in merged_bgs:
    print(f"  {bg.sample_name} / {bg.name}")

merged_bg = [bg for bg in merged_bgs if bg.sample_name == "gfa_merged"][0]
w = merged_bg.plot()
w.show_path()
w

### 3.6 `update_with_gaf`

Aligns reads from a GAF file onto an existing graph, recording observed paths as new samples.

In [ ]:
gaf_repo = gen.Repository(str(WORK_DIR / "gaf_demo"))
gaf_repo.import_fasta(str(FX / "chr22_het.fa"), sample="ref")
gaf_repo.import_gfa(str(FX / "chr22_het.gfa"), sample="ref")

# csv: path to the insert-flanks CSV that describes insertions in the GAF
gaf_repo.update_with_gaf(
    str(FX / "chr22_het.gaf"),
    csv=str(FX / "chr22_insert.csv"),
    sample="gaf_sample",
)

bgs = gaf_repo.get_sequence_graphs()
print(f"Sequence graphs after GAF update: {len(bgs)}")
for bg in bgs:
    print(f"  {bg.sample_name} / {bg.name}")

w = bgs[0].plot()
w.show_path()
w

### 3.7 `update_with_library` / `update_with_library_files`

Replaces a region with a combinatorial library, producing one path per design.

In [ ]:
lib_update_repo = gen.Repository(str(WORK_DIR / "lib_update"))
lib_update_repo.import_fasta(str(FX / "simple.fa"), sample="ref")

# Build the replacement library inline
parts_list = [
    [gen.SequencePart("p1", "AAAA"), gen.SequencePart("cds1", "ATGATAA")],
    [gen.SequencePart("p2", "TAAT"), gen.SequencePart("cds2", "ATGTTAA")],
]

lib_update_repo.update_with_library(
    sample="ref",
    new_sample_name="lib_updated",
    path_name="m123:5-20",
    parts_list=parts_list,
)

bgs = lib_update_repo.get_sequence_graphs()
print(f"Sequence graphs after library update: {len(bgs)}")
for bg in bgs:
    print(f"  {bg.sample_name} / {bg.name}")

updated_bg = [bg for bg in bgs if bg.sample_name == "lib_updated"][0]
w = updated_bg.plot()
w.show_path()
w

In [ ]:
# Same operation, but reading parts from CSV files
lib_update_repo.update_with_library_files(
    sample="ref",
    new_sample="lib_file_updated",
    path_name="m123:5-20",
    library=str(FX / "combinatorial_design.csv"),
    parts=str(FX / "parts.fa"),
)

bgs = lib_update_repo.get_sequence_graphs()
print(f"Sequence graphs after library-files update: {len(bgs)}")

---
## 4. Graph Operations

Graph operations derive new samples by slicing, chunking, or concatenating existing graphs.

In [ ]:
# chr22_het.fa has several linear ~50bp sequences — good for subgraph and chunk demos
ops_repo = gen.Repository(str(WORK_DIR / "graph_ops"))
ops_repo.import_fasta(str(FX / "chr22_het.fa"), sample="ref")

bgs = ops_repo.get_sequence_graphs()
print(f"Base sequence graphs: {len(bgs)}")
for bg in bgs:
    print(f"  {bg.name} ({bg.sample_name})")

region_name = bgs[0].name

### 4.1 `derive_subgraph`

Extracts the subgraph covering a genomic region into a new sample.

In [ ]:
ops_repo.derive_subgraph(
    sample="ref",
    new_sample="subgraph_sample",
    region=f"{region_name}:0-25",
)

sub_bgs = [bg for bg in ops_repo.get_sequence_graphs() if bg.sample_name == "subgraph_sample"]
print(f"Derived subgraph sequence graphs: {len(sub_bgs)}")
sub_bgs[0].plot()

### 4.2 `derive_chunks`

Splits a block group into fixed-size chunks using explicit breakpoint coordinates.

In [ ]:
# breakpoints is a list of integer coordinates along the path
ops_repo.derive_chunks(
    sample="ref",
    new_sample="chunks_sample",
    region=region_name,
    breakpoints=[25],   # split at position 25 → two chunks [0,25] and [25,end]
)

chunk_bgs = [bg for bg in ops_repo.get_sequence_graphs() if bg.sample_name == "chunks_sample"]
print(f"Chunks created: {len(chunk_bgs)}")
for bg in chunk_bgs:
    print(f"  {bg.name}")

chunk_bgs[0].plot()

### 4.3 `make_stitch`

Joins a comma-separated list of existing block groups end-to-end into a single new block group.

In [ ]:
# Stitch the first three chunks back into one region
stitch_names = ",".join(bg.name for bg in chunk_bgs[:3])

ops_repo.make_stitch(
    sample="chunks_sample",
    new_sample="stitched_sample",
    regions=stitch_names,
    new_region="stitched_region",
)

stitched = [bg for bg in ops_repo.get_sequence_graphs() if bg.sample_name == "stitched_sample"]
print(f"Stitched sequence graphs: {len(stitched)}")
stitched[0].plot()

### 4.4 `stitch` (typed helper)

Same as `make_stitch` but accepts `SequenceGraph` objects directly instead of name strings.

In [24]:
result_bg = ops_repo.stitch(
    bgs=chunk_bgs[:3],
    new_sample="stitch_typed",
    new_region="typed_region",
)

print(f"Stitched: {result_bg.name} in sample '{result_bg.sample_name}'")
result_bg.plot()

Stitched: typed_region in sample 'stitch_typed'
Stitched chunks successfully into new region typed_region in sample stitch_typed.


---
## 5. Using the `collection` parameter

A single `gen` repository can hold data from multiple analyses on the same organism.
Collections let you organise sequence graphs by analysis type while still sharing the same samples —
for example, genomics, transcriptomics, and proteomics data can all live in one repo and
be queried independently or jointly.

Pass `collection=` to any import/export/update call to tag sequence graphs, then retrieve them
with `get_sequence_graphs_by_collection`.

In [ ]:
multi_repo = gen.Repository(str(WORK_DIR / "multi_omics"))

# Each analysis type gets its own collection — same repo, same samples
multi_repo.import_fasta(str(FX / "simple.fa"),   sample="wt", collection="genomics")
multi_repo.import_fasta(str(FX / "parts.fa"),    sample="wt", collection="transcriptomics")
multi_repo.import_fasta(str(FX / "multiseq.fa"), sample="wt", collection="proteomics")
multi_repo.import_fasta(str(FX / "multiseq.fa"), sample="mut", collection="genomics")

genomics      = multi_repo.get_sequence_graphs_by_collection("genomics")
transcriptomics = multi_repo.get_sequence_graphs_by_collection("transcriptomics")
proteomics    = multi_repo.get_sequence_graphs_by_collection("proteomics")

print(f"genomics:        {len(genomics)} sequence graph(s)")
print(f"transcriptomics: {len(transcriptomics)} sequence graph(s)")
print(f"proteomics:      {len(proteomics)} sequence graph(s)")
print()
for bg in genomics:
    print(f"  [genomics] {bg.sample_name} / {bg.name}")